# Milestone 2: Movie Data Acquisition

*Cleaning/Formatting Flat File Source* 

Perform at least 5 data transformation and/or cleansing steps to your flat file data. The below examples are not required - they are just potential transformations you could do. If your data doesn't work for these scenarios, complete different transformations. You can do the same transformation multiple times if needed to clean your data. The goal is a clean dataset at the end of the milestone. 

 

**Examples:** 

- Replace Headers. 
- Format data int a more readable format. 
- Identify outliers and bad data. 
- Find duplicates. 
- Fix casing or inconsistent values. 
- Conduct Fuzzy Matching. 

 

Make sure you clearly number and label each transformation step (Step #1, Step #2, etc.) in your code and describe what it is doing in 1-2 sentences. 

In [ ]:
# Load the necessary libraries.
import pandas as pd
import numpy as np
from scipy import stats

## I'm working with two different .csv files from the DataSet I mentioned on Milestone 1.

In [ ]:
# Load .csv files containing critic reviews and movies information into pandas DataFrames for analysis.
critic_reviews_df = pd.read_csv("rotten_tomatoes_critic_reviews.csv")
movies_df = pd.read_csv("rotten_tomatoes_movies.csv")

# Critics Review DataFrame.

In [ ]:
# Display first few rows of critic reviews DataFrame.
critic_reviews_df.head()

In [ ]:
# Get the original dimensions of the critic reviews DataFrame.
critic_reviews_df.shape

In [ ]:
# Display the data types of each column in the DataFrame to understand the structure of the data.
critic_reviews_df.dtypes

In [ ]:
# STEP 1.
# Create a new DataFrame cleaned_critic_reviews_df from critic_reviews_df
cleaned_critic_reviews_df = critic_reviews_df.copy()

In [ ]:
# STEP 2.
# Format Headers: Removing underscores and capitalize the first letter of each word.
cleaned_critic_reviews_df.columns = cleaned_critic_reviews_df.columns.str.replace("_", " ").str.title()

In [ ]:
# Display first few rows of cleaned_critic_reviews_df to verify change to headers.
cleaned_critic_reviews_df.head()

In [ ]:
# Calculate the total number of missing values in each column of the cleaned_critic_reviews_df.
missing_values = cleaned_critic_reviews_df.isnull().sum()
print(missing_values)

In [ ]:
# STEP 3.
# Fill missing values in "Critic Name" with the placeholder "Unknown Critic."
cleaned_critic_reviews_df["Critic Name"] = cleaned_critic_reviews_df["Critic Name"].fillna("Unknown Critic")

In [ ]:
# STEP 4. 
# Define a function to clean the Review Scores and create a new column named Numeric Review Score.
def clean_review_score(value):
    if pd.isna(value) or value == "":
        return np.nan  # Keep empty cells as NaN.
    elif "/" in value:
        return float(value.split("/")[0])  # Extract numeric value before "/".
    elif isinstance(value, str) and value.replace("-", "").isalpha():  # Check for grades.
        return np.nan  # Treat grades as NaN.
    else:
        return np.nan  # Default case for unexpected formats.

# Apply "clean_review_score" function to "Review Score" column and store the cleaned numeric values in the "Numeric Review Score" column.
cleaned_critic_reviews_df["Numeric Review Score"] = cleaned_critic_reviews_df["Review Score"].apply(clean_review_score)

# Calculate the mean score, ignoring NaN values.
mean_review_score = cleaned_critic_reviews_df["Numeric Review Score"].mean()

# Fill NaN values in the new column with the mean score
cleaned_critic_reviews_df["Numeric Review Score"] = cleaned_critic_reviews_df["Numeric Review Score"].fillna(mean_review_score)

# Display "Mean Review Score".
print("Mean Review Score: ", mean_review_score)

In [ ]:
# STEP 5.
# Fill missing values in Review Content with N/A.
cleaned_critic_reviews_df["Review Content"] = cleaned_critic_reviews_df["Review Content"].fillna("N/A")

In [ ]:
# Display the total number of missing values in each column of the DataFrame after cleaning.
# "Numeric Review Score" column is a cleaned version of "Review Score" column. 
missing_values = cleaned_critic_reviews_df.isnull().sum()
print(missing_values)

In [ ]:
# STEP 6.
# Convert "Review Date" from string/object to datetime format.
cleaned_critic_reviews_df["Review Date"] = pd.to_datetime(cleaned_critic_reviews_df["Review Date"], errors = "coerce")

In [ ]:
# Check unique values from the Review Type column to identify distinct review categories.
unique_values = cleaned_critic_reviews_df["Review Type"].unique()
print("Unique values in Review Type: ", unique_values)

In [ ]:
# STEP 7.
# Check for duplicate reviews based on "Rotten Tomatoes Link" and "Critic Name."
duplicate_reviews = cleaned_critic_reviews_df.duplicated(subset = ["Rotten Tomatoes Link", "Critic Name"], keep = False)

# # Display duplicate rows, if there are any.
duplicate_entries = cleaned_critic_reviews_df[duplicate_reviews]
print(duplicate_entries)

# Remove duplicates, keeping the first occurrence.
cleaned_critic_reviews_df.drop_duplicates(subset = ["Rotten Tomatoes Link", "Critic Name"], inplace = True)

In [ ]:
# Get the shape of cleaned_critic_reviews_df.
cleaned_critic_reviews_df.shape

In [ ]:
# Display cleaned_critic_reviews_df types of each column in the DataFrame.
cleaned_critic_reviews_df.dtypes

In [ ]:
# Display first few rows of the cleaned_critic_reviews_df DataFrame.
cleaned_critic_reviews_df.head()

In [ ]:
# Display the cleaned DataFrame to review the changes made during the cleaning process.
print(cleaned_critic_reviews_df)

In [ ]:
# Calculate the Z-scores for 'Numeric Review Score.'
cleaned_critic_reviews_df["z_score"] = stats.zscore(cleaned_critic_reviews_df["Numeric Review Score"])

# Identify outliers (e.g., Z-score > 3 or < -3).
outliers = cleaned_critic_reviews_df[(cleaned_critic_reviews_df["z_score"] > 3) | (cleaned_critic_reviews_df["z_score"] < -3)]

# Display the outliers.
print(outliers)

- A total of 6005 rows were analyzed, and outliers were identified as those with Z-scores greater than 3 or less than -3. These include scores like 83.0, 92.0, and higher, indicating they are exceptionally high. 
- The outliers suggest unusually positive reviews and may represent specific critics or contexts contributing to these high scores.

- Significant outliers in the review scores, warrant further investigation to understand their implications. ns. 

In [ ]:
# Check for unexpected values in categorical columns.
print(cleaned_critic_reviews_df["Critic Name"].value_counts())
print(cleaned_critic_reviews_df["Review Type"].value_counts())
print(cleaned_critic_reviews_df["Review Content"].value_counts())

**Critic Name:**
The dataset shows a significant concentration of reviews from a few critics, with “Unknown Critic” contributing **7121 reviews**. This indicates potential bias and over-reliance on certain voices. 

**Review Type:**
A notable predominance of “Fresh” reviews **634968** compared to “Rotten” reviews **361502** suggests a generally positive bias in the dataset. 

**Review Content:**
Many entries are marked as `N/A` **53352**, indicating missing review content. This raises concerns about data completeness.
Other entries vary widely, with some unique reviews only appearing once, pointing to a diverse range of opinions but limited utility due to high `N/A` counts.

The analysis highlights potential biases in critic representation and a positive skew in review types while also identifying significant gaps in the review content that could affect further analyses. 

In [ ]:
# Check if 'Review Date' has any invalid dates or outliers
print(cleaned_critic_reviews_df["Review Date"].min(), cleaned_critic_reviews_df["Review Date"].max())

- The dataset shows a review date range from **January 1, 1800** to **October 29, 2020.** 
- The early date of 1800 is likely an invalid entry, as it predates modern film criticism and raises concerns about data accuracy. 
- This suggests potential data entry errors or inconsistencies that need to be addressed. 

# Movies DataFrame.

In [ ]:
movies_df.head()

In [ ]:
# Get the original dimensions of the movies DataFrame.
movies_df.shape

In [ ]:
# Display the data types of each column in the DataFrame to understand the structure of the data.
movies_df.dtypes

In [ ]:
# STEP 1.
# Create a new DataFrame cleaned_movies_df from movies_df.
cleaned_movies_df = movies_df.copy()

In [ ]:
# STEP 2.
# Format Headers: Removing underscores and capitalize the first letter of each word.
cleaned_movies_df.columns = cleaned_movies_df.columns.str.replace("_", " ").str.title()

In [ ]:
# Display first few rows of cleaned_critic_reviews_df to verify change to headers.
cleaned_movies_df.head()

In [ ]:
# Calculate the total number of missing values in each column of the "cleaned_movies_df."
movies_missing_values = cleaned_movies_df.isnull().sum()
print(movies_missing_values)

In [ ]:
# STEPS 3 - 11.
# Fill missing values in categorical (object) columns with the appropriate placeholder.
cleaned_movies_df["Movie Info"] = cleaned_movies_df["Movie Info"].fillna("Unknown Info")
cleaned_movies_df["Critics Consensus"] = cleaned_movies_df["Critics Consensus"].fillna("No Consensus")
cleaned_movies_df["Genres"] = cleaned_movies_df["Genres"].fillna("Unknown Genre")
cleaned_movies_df["Directors"] = cleaned_movies_df["Directors"].fillna("Unknown Director")
cleaned_movies_df["Authors"] = cleaned_movies_df["Authors"].fillna("Unknown Author")
cleaned_movies_df["Actors"] = cleaned_movies_df["Actors"].fillna("Unknown Actors")
cleaned_movies_df["Production Company"] = cleaned_movies_df["Production Company"].fillna("Unknown Production Company")
cleaned_movies_df["Tomatometer Status"] = cleaned_movies_df["Tomatometer Status"].fillna("Unknown Status")
cleaned_movies_df["Audience Status"] = cleaned_movies_df["Audience Status"].fillna("Unknown Audience Status")

In [ ]:
# STEPS 12 - 15.
# Fill missing numerical (float64) columns with the median.
cleaned_movies_df["Tomatometer Rating"] = cleaned_movies_df["Tomatometer Rating"].fillna(cleaned_movies_df["Tomatometer Rating"].median())
cleaned_movies_df["Tomatometer Count"] = cleaned_movies_df["Tomatometer Count"].fillna(cleaned_movies_df["Tomatometer Count"].median())
cleaned_movies_df["Audience Rating"] = cleaned_movies_df["Audience Rating"].fillna(cleaned_movies_df["Audience Rating"].median())
cleaned_movies_df["Audience Count"] = cleaned_movies_df["Audience Count"].fillna(cleaned_movies_df["Audience Count"].median())

In [ ]:
# STEPS 16 - 17.
# Convert "Original Release Date" and "Streaming Release Date" to datetime format
cleaned_movies_df["Original Release Date"] = pd.to_datetime(cleaned_movies_df["Original Release Date"], errors = "coerce")
cleaned_movies_df["Streaming Release Date"] = pd.to_datetime(cleaned_movies_df["Streaming Release Date"], errors = "coerce")

In [ ]:
# STEPS 18 - 19.
# Fill missing "Original Release Date" and "Streaming Release Date" with a placeholder date
cleaned_movies_df["Original Release Date"] = cleaned_movies_df["Original Release Date"].fillna(pd.to_datetime("1800-01-01"))
cleaned_movies_df["Streaming Release Date"] = cleaned_movies_df["Streaming Release Date"].fillna(pd.to_datetime("1800-01-01"))

In [ ]:
# Step 20.
# Fill missing values in 'Runtime' with 0 as a placeholder.
cleaned_movies_df["Runtime"] = cleaned_movies_df["Runtime"].fillna(0)

In [ ]:
# Recheck missing values in each column of the "cleaned_movies_df."
movies_missing_values = cleaned_movies_df.isnull().sum()
print(movies_missing_values)

In [ ]:
# Check unique values 'Tomatometer Status' and 'Audience Status' to identify distinct review categories.
print(cleaned_movies_df['Tomatometer Status'].unique())
print(cleaned_movies_df['Audience Status'].unique())


In [ ]:
# Check for duplicates based on "Rotten Tomatoes Link"
duplicates_rotten_tomatoes = cleaned_movies_df.duplicated(subset = "Rotten Tomatoes Link", keep = False)

# Check for duplicates based on "Movie Title".
duplicates_movie_title = cleaned_movies_df.duplicated(subset = "Movie Title", keep = False)

# Display duplicate rows based on "Rotten Tomatoes Link" and "Movie Title"
print(cleaned_movies_df[duplicates_rotten_tomatoes])
print(cleaned_movies_df[duplicates_movie_title])

In [ ]:
# Steps 21 - 22.
# Remove duplicates based on "Rotten Tomatoes Link" while keeping the first occurrence.
cleaned_movies_df.drop_duplicates(subset = "Rotten Tomatoes Link", keep = "first", inplace = True)

# Remove duplicates based on "Movie Title"
cleaned_movies_df.drop_duplicates(subset = "Movie Title", keep = "first", inplace = True)

In [ ]:
# Check for remaining duplicates in 'rotten_tomatoes_link' and 'movie_title'
print(f"Remaining duplicates in 'Rotten Tomatoes Link': {cleaned_movies_df.duplicated(subset = 'Rotten Tomatoes Link').sum()}")
print(f"Remaining duplicates in 'Movie Title': {cleaned_movies_df.duplicated(subset = 'Movie Title').sum()}")

In [ ]:
# Get the shape of cleaned_movies_df.
cleaned_movies_df.shape

In [ ]:
# Display cleaned_movies_df types of each column in the DataFrame.
cleaned_movies_df.dtypes

In [ ]:
# Display first few rows of the cleaned_movies_df.
cleaned_movies_df.head()

In [ ]:
# Display the cleaned DataFrame to review the changes made during the cleaning process.
print(cleaned_movies_df)

In [ ]:
# Defining a list of numeric (float64) columns for analysis.
numeric_columns = ["Tomatometer Rating", "Audience Rating", "Tomatometer Count", "Audience Count"]

# Loop through each column to identify outliers using Z-score
for column in numeric_columns:
    # Calculate the Z-scores
    cleaned_movies_df[f'z_score_{column}'] = stats.zscore(cleaned_movies_df[column])
    
    # Identify outliers with Z-scores > 3 or < -3
    outliers = cleaned_movies_df[(cleaned_movies_df[f'z_score_{column}'] > 3) | (cleaned_movies_df[f'z_score_{column}'] < -3)]
    
    print(f"Outliers in {column}:")
    print(outliers[[column, f'z_score_{column}']])

**Tomatometer Rating:** No outliers were found, indicating a normal distribution of ratings. 

**Audience Rating:** No outliers were identified, suggesting consistent ratings across the dataset. 

**Tomatometer Count**: Several outliers were detected, ranging from approximately 3.0 to 4.35, indicating that some films received significantly more critic reviews, likely due to popularity or acclaim. 

**Audience Count:** Multiple outliers were found, with extremely high (exceeding 18) review counts for certain films, reflecting strong audience engagement. 

In [ ]:
# Check for unique values in categorical columns
print(cleaned_movies_df['Genres'].value_counts())
print(cleaned_movies_df['Directors'].value_counts())

**Genres:** 
- Drama (1828) and Comedy (1233) dominate, showing a strong audience preference. 
- Comedy and Drama (843) indicates a blend of these styles. 
- Genres like Drama, Mystery & Suspense (702) and Art House & International, Drama (573) highlight varied storytelling approaches. 
- There are 1090 unique genre combinations in the dataset. 

**Directors**: 
- Unknown Director (189) points to many films lacking prominent names, often from independent projects. 
- Woody Allen and Clint Eastwood (36 each) are significant figures in the dataset. 
- Alfred Hitchcock (34) reflects the legacy of classic directors. 
- The dataset includes 8748 unique directors, showcasing a wide range of talent. 

In [ ]:
# Check for missing values in the dataset.
missing_values = cleaned_movies_df.isnull().sum()
print("Missing values in each column:")
print(missing_values)

# Check for unusual zero values in important columns.
invalid_runtime = cleaned_movies_df[cleaned_movies_df['Runtime'] == 0]
print("Movies with runtime of 0 (possible invalid data):")
print(invalid_runtime[['Movie Title', 'Runtime']])

No missing values are in any column of the “cleaned_movies_df" DataFrame, indicating complete and reliable data for analysis. 

There are 302 movies with a runtime of 0, which is likely an error, as all films should have a defined duration. This raises concerns about data accuracy. 

Multiple entries with a runtime of 0 suggest potential data entry issues that require further investigation. 

Correct or exclude the zero-runtime entries to ensure data integrity. 

# Human readable dataset after all transformations for "cleaned_critic_reviews_df."

In [ ]:
# Display the cleaned cleaned_critic_reviews_df DataFrame to review the changes made during the cleaning process.
print(cleaned_critic_reviews_df)

# Human readable dataset after all transformations for "cleaned_movies_df."

In [ ]:
# Display the cleaned_movies_df DataFrame to review the changes made during the cleaning process.
print(cleaned_movies_df)

# Ethical implications of data wrangling specific to my datasources.

    In the process of cleaning and transforming the “leaned_critic_reviews_df” and “cleaned_movies_df” datasets, several modifications were made, including handling missing values, standardizing categorical columns, filling null values with placeholders, and identifying/removing duplicates. These changes could introduce risks related to data integrity, such as altering the intended meaning of reviews or movie ratings, especially when assumptions like filling missing scores or dates were made. Since movie data and reviews are often public, fewer legal concerns may exist. Still, it is critical to comply with privacy and data protection regulations, especially if the personal information of critics is involved. Data credibility is sourced from established platforms like Rotten Tomatoes, but ethical acquisition and transparency in data handling remain important. To mitigate potential ethical concerns, it is essential to document all transformations, ensure that assumptions do not bias the analysis, and handle public reviews and movie metadata in a way that respects the source and original context. 

In [ ]:
cleaned_critic_reviews_df.head()

In [ ]:
cleaned_movies_df.head()

In [ ]:
cleaned_critic_reviews_df.to_csv("rt_cleaned_critic_reviews_df.csv", index=False)

In [ ]:
cleaned_movies_df.to_csv("rt_cleaned_movies_df.csv", index=False)